# Exploration de l'API TED (marchés publics européens)

**Projet :** Plateforme d'automatisation de la prospection (veille AO / RFP / CFP)
**Module :** 1 - Collecte
**Source étudiée :** TED (Tenders Electronic Daily), le journal officiel des marchés publics de l'UE.

## Pourquoi cette source

La BCEAO publie très peu de cybersécurité. Je cherche une source avec du volume et de vraies
opportunités cyber. TED est intéressant pour trois raisons : c'est une **API officielle** (donc
des données structurées, pas du HTML à scraper), le **volume est énorme** (toute l'Europe), et
je peux **filtrer** par mots-clés, pays et catégorie directement dans la requête.

## Objectif du carnet

Comprendre la structure réelle des données AVANT d'écrire le collecteur : quels champs existent,
lesquels sont fiables, comment repérer la cybersécurité, et quelles valeurs sont manquantes ou
aberrantes. Je documente tout, comme pour la BCEAO.

## 1. L'API TED

- Endpoint : `POST https://api.ted.europa.eu/v3/notices/search`
- Accès **anonyme** en lecture (pas de clé API nécessaire).
- On envoie une requête JSON avec : `query` (les filtres), `fields` (les champs voulus),
  `limit`, `page`, `scope`.
- Je respecte la *fair usage policy* : pas de grosses boucles en phase de test.

In [1]:
import requests
import json
from collections import Counter
from datetime import date

URL = "https://api.ted.europa.eu/v3/notices/search"

## 2. Découvrir les champs disponibles

L'API a énormément de champs et je ne les connais pas. Petite astuce : je demande volontairement
un champ qui n'existe pas (`__bidon__`). L'API renvoie alors un message d'erreur qui **liste tous
les champs valides**. C'est plus rapide que de fouiller la documentation, et ça me donne la
vérité directement depuis l'API.

In [2]:
body = {"query": "publication-date>=today(-3)", "fields": ["__bidon__"],
        "limit": 1, "page": 1, "scope": "ACTIVE"}
r = requests.post(URL, json=body, timeout=30)

message = r.json()["message"]
liste_brute = message.split("supported values are:")[1].rstrip(")").strip()
champs = sorted(c.strip() for c in liste_brute.split(",") if c.strip())
print("Nombre total de champs proposés par l'API :", len(champs))

Nombre total de champs proposés par l'API : 1830


Il y en a énormément. Beaucoup sont des codes techniques (préfixe `BT-`). Je cherche surtout
les champs dont j'ai besoin pour décrire une opportunité : le titre, la date de publication, la
date limite, le pays, les liens, la catégorie. Je filtre la liste pour les retrouver.

In [3]:
for mot in ["title", "publication-date", "deadline", "buyer-country", "buyer-name", "links", "cpv"]:
    trouves = [c for c in champs if mot in c.lower() and not c.startswith("BT-")][:6]
    print(f"{mot:18} -> {trouves}")

title              -> ['announcement-title', 'contract-title', 'notice-title', 'review-title', 'title-glo', 'title-lot']
publication-date   -> ['notice-preferred-publication-date', 'publication-date']
deadline           -> ['deadline', 'deadline-date-lot', 'deadline-date-part', 'deadline-receipt-answers-date-lot', 'deadline-receipt-answers-time-lot', 'deadline-receipt-expressions-date-lot']
buyer-country      -> ['buyer-country', 'buyer-country-sub']
buyer-name         -> ['buyer-name']
links              -> ['links']
cpv                -> ['classification-cpv']


J'en retiens les champs qui me serviront : `notice-title`, `publication-date`, `deadline`,
`buyer-country`, `buyer-name`, `links`, `classification-cpv`, plus `publication-number` (l'identifiant)
et `notice-type` (pour distinguer une mise en concurrence d'un avis d'attribution).

## 3. Une fonction de recherche

Je me fais une petite fonction utilitaire pour ne pas réécrire la requête à chaque fois.

In [4]:
CHAMPS = ["publication-number", "notice-title", "buyer-name", "buyer-country",
          "publication-date", "deadline", "notice-type", "classification-cpv", "links"]

def chercher(query, limit=50, scope="ACTIVE"):
    body = {"query": query, "fields": CHAMPS, "limit": limit, "page": 1, "scope": scope}
    r = requests.post(URL, json=body, timeout=40)
    r.raise_for_status()
    return r.json()

## 4. Premier appel : chercher des annonces de cybersécurité

Je cherche par mots-clés dans le titre (cybersécurité, tests d'intrusion, ISO 27001, etc.) et je
me limite aux **mises en concurrence** (`notice-type=cn-standard`), c'est-à-dire des appels
d'offres ouverts, pas des avis d'attribution déjà clôturés.

In [5]:
REQUETE_CYBER = (
    '(notice-title ~ "cybersecurity" OR notice-title ~ "penetration" '
    'OR notice-title ~ "ISO 27001" OR notice-title ~ "security audit" '
    'OR notice-title ~ "vulnerability" OR notice-title ~ "pentest") '
    'AND notice-type=cn-standard SORT BY publication-date DESC'
)

data = chercher(REQUETE_CYBER, limit=100)
print("Clés de la réponse :", list(data.keys()))
print("Total correspondant :", data["totalNoticeCount"])
print("Annonces récupérées :", len(data["notices"]))

Clés de la réponse : ['notices', 'totalNoticeCount', 'iterationNextToken', 'timedOut']
Total correspondant : 128
Annonces récupérées : 100


Le total est largement supérieur à ce que la BCEAO offrait en cybersécurité. La source est
clairement plus riche pour mon sujet.

## 5. Regarder une annonce en détail

Avant tout traitement, je regarde la structure brute d'une annonce. C'est là que je découvre les
formats particuliers : titres multilingues, dates avec fuseau horaire, champs sous forme de listes.

In [6]:
annonce = data["notices"][0]
print(json.dumps(annonce, indent=2, ensure_ascii=False)[:1200])

{
  "notice-type": "cn-standard",
  "publication-number": "418672-2026",
  "classification-cpv": [
    "48000000",
    "48517000",
    "48730000",
    "72263000",
    "72267000",
    "48000000",
    "48517000",
    "48730000",
    "72263000",
    "72267000"
  ],
  "buyer-name": {
    "deu": [
      "BWI GmbH"
    ]
  },
  "publication-date": "2026-06-18+02:00",
  "links": {
    "xml": {
      "MUL": "https://ted.europa.eu/en/notice/418672-2026/xml"
    },
    "pdf": {
      "BUL": "https://ted.europa.eu/bg/notice/418672-2026/pdf",
      "SPA": "https://ted.europa.eu/es/notice/418672-2026/pdf",
      "CES": "https://ted.europa.eu/cs/notice/418672-2026/pdf",
      "DAN": "https://ted.europa.eu/da/notice/418672-2026/pdf",
      "DEU": "https://ted.europa.eu/de/notice/418672-2026/pdf",
      "EST": "https://ted.europa.eu/et/notice/418672-2026/pdf",
      "ELL": "https://ted.europa.eu/el/notice/418672-2026/pdf",
      "ENG": "https://ted.europa.eu/en/notice/418672-2026/pdf",
      "FRA": "h

Je remarque déjà plusieurs choses : le `notice-title` est un dictionnaire avec une entrée
par langue, les dates ont un fuseau horaire, et `classification-cpv` est une liste (parfois avec
des codes répétés). Je vais vérifier tout ça proprement dans le contrôle qualité.

## 6. Quels codes CPV correspondent à la cybersécurité ?

Le CPV (Common Procurement Vocabulary) est le système de codes qui classe les marchés publics
européens. Je me demandais s'il existait UN code pour la cybersécurité. Plutôt que de le
supposer, je regarde quels codes CPV apparaissent réellement sur mes annonces cyber.

In [7]:
compteur_cpv = Counter()
for n in data["notices"]:
    for code in set(n.get("classification-cpv", [])):
        compteur_cpv[code] += 1

print("Codes CPV les plus fréquents sur les annonces cyber :\n")
for code, nb in compteur_cpv.most_common(12):
    print(f"  {code} : {nb} annonces")

Codes CPV les plus fréquents sur les annonces cyber :

  72000000 : 12 annonces
  71300000 : 9 annonces
  48000000 : 7 annonces
  48730000 : 7 annonces
  79417000 : 7 annonces
  71000000 : 6 annonces
  71330000 : 6 annonces
  71317000 : 5 annonces
  71335000 : 5 annonces
  72820000 : 5 annonces
  79311000 : 5 annonces
  72220000 : 4 annonces


**Conclusion importante :** il n'existe pas UN code CPV unique pour la cybersécurité. Elle
est répartie entre des codes commençant par `72` (services informatiques) et `48` (logiciels),
selon la façon dont l'acheteur a classé son marché. Donc filtrer uniquement par CPV laisserait
passer beaucoup de bruit (et raterait des annonces mal classées). Je préfère filtrer par
**mots-clés dans le titre**, et laisser ensuite le scoring IA (module 2) faire le tri fin entre
ce qui est vraiment de la cybersécurité et ce qui ne l'est pas.

## 7. Contrôle qualité des données

C'est l'étape clé avant de coder le collecteur. Je vérifie, sur mon échantillon réel :
la complétude des champs, les doublons, les langues des titres, la date limite et son format,
et la structure des liens.

In [8]:
notices = data["notices"]
n = len(notices)
print(f"Échantillon analysé : {n} annonces\n")

print("a) COMPLÉTUDE (champs réellement présents) :")
for champ in CHAMPS:
    presents = sum(1 for x in notices if x.get(champ) not in (None, "", [], {}))
    print(f"   {champ:20} : {presents}/{n}  ({100*presents//n}%)")

Échantillon analysé : 100 annonces

a) COMPLÉTUDE (champs réellement présents) :
   publication-number   : 100/100  (100%)
   notice-title         : 100/100  (100%)
   buyer-name           : 100/100  (100%)
   buyer-country        : 100/100  (100%)
   publication-date     : 100/100  (100%)
   deadline             : 16/100  (16%)
   notice-type          : 100/100  (100%)
   classification-cpv   : 100/100  (100%)
   links                : 100/100  (100%)


In [9]:
print("b) DOUBLONS sur publication-number :")
nums = [x.get("publication-number") for x in notices]
print(f"   {len(nums)} numéros, {len(set(nums))} uniques -> {len(nums)-len(set(nums))} doublon(s)")

b) DOUBLONS sur publication-number :
   100 numéros, 100 uniques -> 0 doublon(s)


In [10]:
print("c) LANGUES des titres :")
sans_anglais = sum(1 for x in notices
                   if isinstance(x.get("notice-title"), dict) and "eng" not in x["notice-title"])
print(f"   annonces sans titre anglais : {sans_anglais}/{n}")
print("   -> si 0, je peux prendre l'anglais comme langue par défaut sans risque.")

c) LANGUES des titres :
   annonces sans titre anglais : 0/100
   -> si 0, je peux prendre l'anglais comme langue par défaut sans risque.


In [11]:
print("d) DATE LIMITE (deadline) : présence et format")
avec_dl = [x for x in notices if x.get("deadline")]
print(f"   présente dans : {len(avec_dl)}/{n} annonces")
if avec_dl:
    print("   exemples de format :", [x["deadline"] for x in avec_dl[:2]])
    auj = date.today().isoformat()
    def premiere(d):
        return d[0] if isinstance(d, list) else d
    passees = sum(1 for x in avec_dl if str(premiere(x["deadline"]))[:10] < auj)
    print(f"   déjà passées alors que scope=ACTIVE : {passees}")

d) DATE LIMITE (deadline) : présence et format
   présente dans : 16/100 annonces
   exemples de format : [['2026-06-29T15:00:00+01:00'], ['2026-06-29T23:59:59+03:00', '2026-06-29T23:59:59+03:00', '2026-06-29T23:59:59+03:00', '2026-06-29T23:59:59+03:00', '2026-06-29T23:59:59+03:00', '2026-06-29T23:59:59+03:00', '2026-06-29T23:59:59+03:00', '2026-06-29T23:59:59+03:00', '2026-06-29T23:59:59+03:00']]
   déjà passées alors que scope=ACTIVE : 13


In [12]:
print("e) STRUCTURE des liens :")
ex = next((x.get("links") for x in notices if x.get("links")), None)
print("  ", json.dumps(ex, ensure_ascii=False)[:180], "...")
# Plutôt que de fouiller cette structure, je reconstruirai l'URL depuis le numéro :
num = notices[0]["publication-number"]
print("\n   URL reconstruite depuis le numéro :",
      f"https://ted.europa.eu/fr/notice/{num}")

e) STRUCTURE des liens :
   {"xml": {"MUL": "https://ted.europa.eu/en/notice/418672-2026/xml"}, "pdf": {"BUL": "https://ted.europa.eu/bg/notice/418672-2026/pdf", "SPA": "https://ted.europa.eu/es/notice/418672 ...

   URL reconstruite depuis le numéro : https://ted.europa.eu/fr/notice/418672-2026


### Ce que le contrôle qualité m'apprend

- **Champs fiables (100 % présents)** : `publication-number`, `notice-title`, `buyer-name`,
  `buyer-country`, `publication-date`, `classification-cpv`, `links`.
- **`publication-number` est unique** (0 doublon) : ce sera ma clé de déduplication.
- **Les titres sont toujours disponibles en anglais** : je prends `eng` par défaut.
- **La date limite (`deadline`) est souvent absente** (présente dans une minorité d'annonces).
  Je ne peux donc pas m'en servir comme critère principal de "encore ouvert". Heureusement,
  `scope=ACTIVE` dans ma requête garantit déjà que l'annonce est active. La date limite sera
  donc un bonus si présente, pas un champ vital. Quand elle existe, son format est
  `2026-06-29T15:00:00+01:00` (parfois une liste), à normaliser en `2026-06-29`.
- **Les liens** ont une structure complexe (XML/PDF par langue). Plutôt que de la fouiller, je
  reconstruis l'URL lisible de l'annonce à partir du numéro : `https://ted.europa.eu/fr/notice/<num>`.

## 8. Décisions de conception pour le collecteur TED

D'après cette exploration, voici comment j'écrirai `collecter_ted()` pour produire le même format
que ma collecte BCEAO (afin que les deux sources alimentent la même base) :

| Champ de ma base | Source TED |
|---|---|
| `cle_unique` | `ted::<publication-number>` (unique, toujours présent) |
| `intitule` | `notice-title["eng"]` (toujours disponible) |
| `source` | `"TED"` |
| `buyer` / organisme | `buyer-name` |
| pays | `buyer-country` (utile pour le périmètre géographique) |
| `date_publication` | `publication-date` (normalisée AAAA-MM-JJ) |
| `delai_soumission` | `deadline` si présent (sinon `null`), normalisé |
| `lien` | `https://ted.europa.eu/fr/notice/<numéro>` |
| `statut_source` | `"en cours"` (garanti par `scope=ACTIVE`) |
| `cpv` | `classification-cpv` (pour info / filtrage futur) |

Le filtrage cybersécurité se fait par mots-clés à la collecte, puis le scoring IA tranche.

## 9. Prochaines étapes

1. Écrire la fonction `collecter_ted()` selon le tableau ci-dessus, et la tester sur de vraies données.
2. L'intégrer dans `collecte_et_scoring.py`, à côté de `collecter_bceao()`, pour que les deux
   sources remplissent la même collection MongoDB.
3. Vérifier que le scoring (module 2) repère bien les vraies annonces cyber parmi les résultats TED.
4. Plus tard : gérer la pagination TED (l'API renvoie un `iterationNextToken`) si je veux plus
   que les 100 premières annonces par exécution.